# Does UK financial news sentiment improve out-of-sample equity premium forecasts?

**Empirical pipeline. MSc Finance Analytics dissertation.**

=============================================================================
 UK EQUITY PREMIUM PREDICTABILITY WITH GUARDIAN NEWS SENTIMENT
 Core library -- shared by the notebook and the command-line script.
=============================================================================

Research question
-----------------
Does textual sentiment extracted from UK financial news improve the
out-of-sample predictability of the UK equity premium, beyond what is
achievable using traditional predictors alone?

Design
------
  benchmark  : expanding historical mean of the premium      (Welch-Goyal 2008)
  baseline   : dy, term_spread, short_rate, infl, rvol
  augmented  : baseline + monthly news sentiment index
The baseline predictors are strictly nested inside the augmented set, so any
difference in forecast accuracy is attributable to the sentiment signal alone.

PANDAS COMPATIBILITY
--------------------
This module never uses the resample frequency aliases "M" or "ME", which
changed meaning between pandas 2.1 and 2.2 and are a common source of
breakage.  All monthly aggregation goes through PeriodIndex with freq="M",
whose meaning has been stable across every 1.x, 2.x and 3.x release.  Tested
on pandas 2.1.4 and pandas 3.0.2 with byte-identical output.

---

## What this notebook does

| Step | Cell group | Output |
|---|---|---|
| 1 | Setup | check environment, set paths |
| 2 | Library | all functions, no side effects |
| 3 | Panel | `panel.csv`, Table 5.1 |
| 4 | Sentiment | `sentiment_monthly_finbert.csv`, `sentiment_monthly_lm.csv` |
| 5 | Forecasts | `forecasts.csv` |
| 6 | Statistical evaluation | Table 5.2 |
| 7 | Economic evaluation | Table 5.3 |
| 8 | Robustness | Tables 5.4 to 5.6, look-ahead test |
| 9 | Figures and export | three PNGs, all CSVs |

## Before you start

Put these six files in the folder named in `DATA_DIR` below:

```
FTSE All-Share Total Return GBP End Of Day Historical Results Price Data.csv
FTSE_dividend_yield.csv
Bank Rates.csv
10-year gilt yield.csv
CPI.csv
News Articles.csv
```

`lm_words.json` should sit there too if you want the Loughran-McDonald
comparison without installing anything.

**Run order.** Run every cell in Section 2 once (they only define things), then
work down. The only slow cell is the FinBERT scorer in Section 4; everything
else finishes in minutes. The FinBERT cell is safe to interrupt and re-run.

## Pandas compatibility

The resample frequency alias changed from `M` to `ME` between pandas 2.1 and
2.2, and code written for one raises on the other. This notebook never uses
either. All monthly aggregation goes through `PeriodIndex(freq="M")`, whose
meaning has been stable across every pandas release. Verified to give
byte-identical results on pandas 2.1.4, 2.3.3 and 3.0.2, with `FutureWarning`
and `DeprecationWarning` escalated to errors.

---
## 1. Setup

In [ ]:
# Colab only. Skip locally if these are already installed.
# torch and transformers are needed ONLY by the FinBERT cell in Section 4.
# !pip -q install pandas numpy scikit-learn matplotlib
# !pip -q install transformers torch          # FinBERT
# !pip -q install pysentiment2                # optional, only if lm_words.json is missing

In [ ]:
import os, sys, platform

# ---- EDIT THESE TWO LINES ---------------------------------------------------
DATA_DIR = "data"        # folder holding the six raw CSV files
OUT_DIR  = "output"      # everything this notebook produces
# ----------------------------------------------------------------------------

# On Colab, mount Drive and point DATA_DIR at your folder there instead:
#   from google.colab import drive; drive.mount("/content/drive")
#   DATA_DIR = "/content/drive/MyDrive/dissertation/data"
#   OUT_DIR  = "/content/drive/MyDrive/dissertation/output"
# Using Drive for OUT_DIR means the FinBERT checkpoints survive a disconnect.

os.environ["UKEP_DATA"] = DATA_DIR
os.environ["UKEP_OUT"]  = OUT_DIR
os.makedirs(OUT_DIR, exist_ok=True)

import numpy as np, pandas as pd
print("python  ", platform.python_version())
print("pandas  ", pd.__version__)
print("numpy   ", np.__version__)
try:
    import sklearn; print("sklearn ", sklearn.__version__)
except ImportError:
    print("sklearn  MISSING -- install it before Section 5")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

print("\ndata folder:", os.path.abspath(DATA_DIR))
if os.path.isdir(DATA_DIR):
    for f in sorted(os.listdir(DATA_DIR)):
        print(f"   {f}  ({os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6:.1f} MB)")
else:
    print("   DOES NOT EXIST -- fix DATA_DIR above")

---
## 2. Library

These cells only define things. Run them all once, top to bottom, then move on.
They are the exact contents of `ukep_core.py`, so the notebook and the
command-line script cannot drift apart.

In [ ]:
from __future__ import annotations

import json
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

**2.1 Configuration** &mdash; Configuration. Every choice a reader might question lives here.

In [ ]:
# --- Where things live -------------------------------------------------------
# Point DATA_DIR at the folder containing the six raw CSV files.
DATA_DIR = Path(os.environ.get("UKEP_DATA", "data"))
OUT_DIR = Path(os.environ.get("UKEP_OUT", "output"))

RAW = {
    "tr_daily": "FTSE All-Share Total Return GBP End Of Day Historical Results Price Data.csv",
    "dy": "FTSE_dividend_yield.csv",
    "rates": "Bank Rates.csv",
    "gilt": "10-year gilt yield.csv",
    "cpi": "CPI.csv",
    "articles": "News Articles.csv",
}

# --- Sample ------------------------------------------------------------------
# The dividend yield is a trailing-twelve-month construction, so it only
# becomes available twelve months after the index series begins.  That fixes
# the panel start at 2001-01 rather than the 2000-01 stated in the draft.
SAMPLE_START = "2001-01"
SAMPLE_END = "2025-12"

# First month for which an out-of-sample forecast is produced.  Everything
# before it is the initial training window.  120 months of training data is
# the conventional minimum in this literature.
OOS_START = os.environ.get("UKEP_OOS_START", "2011-01")

# --- Variables ---------------------------------------------------------------
TARGET = "eqp"
BASELINE_PREDICTORS = ["dy", "term_spread", "short_rate", "infl", "rvol"]
SENTIMENT_COL = "sent"
AUGMENTED_PREDICTORS = BASELINE_PREDICTORS + [SENTIMENT_COL]

# --- Risk-free rate ----------------------------------------------------------
# IUDSOIA = SONIA, a realised overnight market rate.  IUDBEDR = Bank Rate, the
# administered policy rate, used as a robustness check.
RF_SERIES = "IUDSOIA"
RF_SERIES_ALT = "IUDBEDR"

# --- Models ------------------------------------------------------------------
# "ols" is the classic linear predictive regression of the Welch-Goyal and
# Campbell-Thompson literature. Keeping it alongside the machine-learning
# methods is what lets the study separate the contribution of the PREDICTORS
# from the contribution of MODEL FLEXIBILITY, as promised in Section 4.2.
MODELS = ["ols", "enet", "rf", "gbrt"]
ADD_COMBINATION = True          # equal-weighted average of the three models
RETUNE_EVERY = int(os.environ.get("UKEP_RETUNE", 12))
CV_FOLDS = 5
SEED = 42

# --- Campbell-Thompson restriction -------------------------------------------
# Floor the equity premium forecast at zero.  Applied to model forecasts only:
# clipping the benchmark as well would change the denominator of the
# out-of-sample R-squared and is not what Campbell and Thompson (2008) do.
IMPOSE_CT_RESTRICTION = True
CT_ALSO_ON_BENCHMARK = False

# --- Economic evaluation -----------------------------------------------------
GAMMA = 3.0
W_MIN, W_MAX = 0.0, 1.5
VOL_WINDOW = 60                 # months in the rolling variance forecast
VOL_MIN_PERIODS = 36
TRANSACTION_COST = 0.0010       # 10 bps, charged on |change in weight|

# --- Sentiment ---------------------------------------------------------------
FINBERT_MODEL = "ProsusAI/finbert"
MAX_PASSAGES_PER_ARTICLE = 10   # cap so one long article cannot dominate
FINBERT_MAX_LEN = 128
FINBERT_BATCH = 128
CHUNK_ROWS = 2000               # articles per checkpoint

**2.2 Pandas-Version-Safe Monthly Helpers** &mdash; Pandas-version-safe monthly helpers. This is what keeps the notebook working on both old and new pandas.

In [ ]:
# Period freq "M" is stable across all pandas versions; the resample aliases
# "M" (<=2.1) and "ME" (>=2.2) are not.  Everything below is expressed in
# monthly Periods and only converted to Timestamps for plotting and export.

def to_month(idx) -> pd.PeriodIndex:
    """Convert any datetime-like index or Series to a monthly PeriodIndex."""
    return pd.PeriodIndex(pd.to_datetime(idx), freq="M")


def month_end_ts(pidx: pd.PeriodIndex) -> pd.DatetimeIndex:
    """Monthly Periods -> Timestamps at the last calendar day of the month."""
    return pd.DatetimeIndex(pidx.to_timestamp(how="end")).normalize()


def monthly_last(s: pd.Series) -> pd.Series:
    """Last observation within each calendar month. Replaces .resample(...).last()."""
    s = s.sort_index()
    g = s.groupby(to_month(s.index))
    out = g.last()
    out.index = pd.PeriodIndex(out.index, freq="M")
    return out.sort_index()


def monthly_mean(s: pd.Series) -> pd.Series:
    s = s.sort_index()
    out = s.groupby(to_month(s.index)).mean()
    out.index = pd.PeriodIndex(out.index, freq="M")
    return out.sort_index()


def monthly_std(s: pd.Series, min_obs: int = 10) -> pd.Series:
    """Within-month sample standard deviation. Replaces .resample(...).std()."""
    s = s.sort_index()
    g = s.groupby(to_month(s.index))
    out = g.std(ddof=1)
    out = out.where(g.size() >= min_obs)
    out.index = pd.PeriodIndex(out.index, freq="M")
    return out.sort_index()

**2.3 Raw Data Loaders** &mdash; Raw data loaders. One per file, each written against that file's actual quirks, each raising loudly rather than guessing.

In [ ]:
# Each loader is written against the exact quirks of the file it reads and
# raises loudly rather than silently producing a wrong series.

def _require(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing input file:\n  {path}\n"
            f"Set UKEP_DATA (currently {DATA_DIR}) to the folder holding the raw CSVs."
        )
    return path


def load_tr_daily() -> pd.Series:
    """
    Daily FTSE All-Share total return index (Investing.com export).

    Quirks handled: UTF-8 BOM in the header, thousands separators inside
    quoted strings, rows in reverse chronological order, day-first dates.
    """
    p = _require(DATA_DIR / RAW["tr_daily"])
    df = pd.read_csv(p, encoding="utf-8-sig")
    if "Date" not in df.columns or "Price" not in df.columns:
        raise ValueError(f"Unexpected columns in {p.name}: {df.columns.tolist()}")
    date = pd.to_datetime(df["Date"], format="%d-%m-%Y", errors="raise")
    price = (df["Price"].astype(str)
             .str.replace(",", "", regex=False)
             .str.strip()
             .astype(float))
    s = pd.Series(price.values, index=date, name="tr_index").sort_index()
    s = s[~s.index.duplicated(keep="last")]
    if s.isna().any():
        raise ValueError("NaNs in the daily total return index.")
    return s


def load_dividend_yield() -> pd.DataFrame:
    """
    Monthly FTSE All-Share price index, total return index and trailing
    dividend yield, as supplied.  Returns a monthly-Period DataFrame.
    """
    p = _require(DATA_DIR / RAW["dy"])
    df = pd.read_csv(p)
    df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="raise")
    df = df.set_index("date").sort_index()
    df.index = to_month(df.index)
    need = {"price_index", "tr_index", "div_yield"}
    if not need.issubset(df.columns):
        raise ValueError(f"{p.name} must contain {sorted(need)}; has {df.columns.tolist()}")
    if df["div_yield"].max() > 1.0:
        raise ValueError("div_yield looks like a percentage; this code expects a decimal.")
    return df[["price_index", "tr_index", "div_yield"]]


def load_rates() -> pd.DataFrame:
    """
    Bank of England daily series: IUDBEDR (Bank Rate) and IUDSOIA (SONIA),
    both in percent per annum.  Dates arrive as '04 Jan 2000'.
    """
    p = _require(DATA_DIR / RAW["rates"])
    df = pd.read_csv(p)
    dcol = "DATE" if "DATE" in df.columns else df.columns[0]
    df[dcol] = pd.to_datetime(df[dcol], format="%d %b %Y", errors="raise")
    df = df.set_index(dcol).sort_index()
    for c in (RF_SERIES, RF_SERIES_ALT):
        if c not in df.columns:
            raise ValueError(f"{p.name} is missing column {c}; has {df.columns.tolist()}")
    return df[[RF_SERIES, RF_SERIES_ALT]].astype(float)


def load_gilt() -> pd.Series:
    """
    FRED IRLTLT01GBM156N, UK 10-year benchmark gilt yield, percent per annum,
    monthly.  FRED stamps monthly observations on the FIRST day of the month;
    the value is a within-month average, so it is treated as information dated
    at that month's end.
    """
    p = _require(DATA_DIR / RAW["gilt"])
    df = pd.read_csv(p)
    dcol = df.columns[0]
    vcol = df.columns[1]
    df[dcol] = pd.to_datetime(df[dcol], errors="raise")
    s = pd.Series(pd.to_numeric(df[vcol], errors="coerce").values,
                  index=to_month(df[dcol]), name="long_yield").sort_index()
    if s.isna().any():
        raise ValueError("NaNs in the gilt yield series.")
    return s


def load_cpi() -> pd.Series:
    """
    ONS series D7BT, CPI all items, 2015=100.  The file carries a metadata
    block, then annual rows, then quarterly rows, then monthly rows.  Only the
    monthly rows ('1988 JAN') are wanted.
    """
    p = _require(DATA_DIR / RAW["cpi"])
    raw = pd.read_csv(p, header=None, names=["label", "value"], dtype=str,
                      engine="python", on_bad_lines="skip")
    pat = re.compile(r"^\s*(\d{4})\s+([A-Z]{3})\s*$")
    keep = raw["label"].astype(str).str.match(pat)
    m = raw.loc[keep].copy()
    if len(m) < 200:
        raise ValueError(f"Only {len(m)} monthly CPI rows parsed from {p.name}; expected 400+.")
    per = pd.PeriodIndex(
        pd.to_datetime(m["label"].str.strip(), format="%Y %b", errors="raise"), freq="M")
    s = pd.Series(pd.to_numeric(m["value"], errors="coerce").values,
                  index=per, name="cpi").sort_index()
    if s.isna().any():
        raise ValueError("NaNs in the CPI series.")
    return s

**2.4 Panel Construction** &mdash; Panel construction and the audit that catches unit errors, sign errors and misalignment before they reach the results.

In [ ]:
def build_panel(rf_series: str = RF_SERIES,
                lag_inflation: bool = True,
                verbose: bool = True) -> pd.DataFrame:
    """
    Assemble the monthly panel indexed by monthly Period.

    Real-time information alignment
    -------------------------------
    Every variable dated month t must be knowable by the close of month t.

      eqp_t          total return on the index during month t, minus the
                     risk-free rate fixed at the START of month t.  Using a
                     within-month or end-of-month rate would put information
                     into the premium that an investor could not have had when
                     the position was opened.
      dy_t           trailing dividends over the month-t closing price.
      short_rate_t   SONIA on the last business day of month t.
      long_yield_t   10-year gilt yield for month t.
      term_spread_t  long_yield_t - short_rate_t.
      infl_t         CPI year-on-year for month t-1.  ONS releases month-t CPI
                     in the middle of month t+1, so the contemporaneous value
                     is NOT in the month-t information set.  Lagging it by one
                     month removes a look-ahead that would otherwise inflate
                     every result in the study.
      rvol_t         annualised standard deviation of daily index returns
                     within month t.
    """
    tr_d = load_tr_daily()
    dyf = load_dividend_yield()
    rates = load_rates()
    gilt = load_gilt()
    cpi = load_cpi()

    # ---- equity side --------------------------------------------------------
    tr_m = monthly_last(tr_d)                       # month-end TR index level
    tr_ret = tr_m.pct_change()                      # total return during month t

    # Cross-check the daily file against the supplied monthly file.
    common = tr_m.index.intersection(dyf.index)
    gap = (tr_m.reindex(common) - dyf["tr_index"].reindex(common)).abs().max()
    if verbose:
        print(f"[check] daily vs monthly TR index, largest gap: {gap:.6f}")
    if gap > 1e-6:
        raise ValueError(
            "The daily and monthly total return series disagree. Resolve before proceeding.")

    daily_ret = tr_d.pct_change()
    rvol = (monthly_std(daily_ret, min_obs=10) * np.sqrt(252)).rename("rvol")

    # ---- risk-free rate -----------------------------------------------------
    rf_eom = monthly_last(rates[rf_series]) / 100.0          # annualised decimal
    rf_start_of_month = (rf_eom.shift(1) / 12.0).rename("rf")  # known at start of t
    short_rate = rf_eom.rename("short_rate")                 # predictor, end of t

    # ---- term spread --------------------------------------------------------
    long_yield = (gilt / 100.0).rename("long_yield")
    term_spread = (long_yield - short_rate).rename("term_spread")

    # ---- inflation ----------------------------------------------------------
    infl_raw = cpi.pct_change(12).rename("infl")
    infl = infl_raw.shift(1) if lag_inflation else infl_raw

    # ---- assemble -----------------------------------------------------------
    panel = pd.concat(
        [tr_ret.rename("tr_ret"),
         dyf["div_yield"].rename("dy"),
         dyf["price_index"].rename("price_index"),
         rf_start_of_month, short_rate, long_yield, term_spread,
         infl.rename("infl"), rvol],
        axis=1)
    panel.index = pd.PeriodIndex(panel.index, freq="M")
    panel = panel.sort_index()

    panel["eqp"] = panel["tr_ret"] - panel["rf"]

    cols = ["eqp", "tr_ret", "rf", "dy", "term_spread", "short_rate",
            "long_yield", "infl", "rvol", "price_index"]
    panel = panel.loc[SAMPLE_START:SAMPLE_END, cols].dropna()
    panel.index.name = "month"

    if verbose:
        print(f"[panel] {len(panel)} months, {panel.index.min()} to {panel.index.max()}")
    _audit_panel(panel, verbose=verbose)
    return panel


def _audit_panel(panel: pd.DataFrame, verbose: bool = True) -> None:
    """Hard sanity checks. These catch sign errors, unit errors and misalignment."""
    problems = []

    ann = panel["eqp"].mean() * 12
    vol = panel["eqp"].std(ddof=1) * np.sqrt(12)
    if not (-0.02 < ann < 0.15):
        problems.append(f"annualised mean premium {ann:.4f} is outside a plausible range")
    if not (0.08 < vol < 0.30):
        problems.append(f"annualised premium volatility {vol:.4f} is implausible")
    if not (0.015 < panel["dy"].mean() < 0.06):
        problems.append(f"mean dividend yield {panel['dy'].mean():.4f} is implausible")
    if not (0.005 < panel["infl"].mean() < 0.06):
        problems.append(f"mean inflation {panel['infl'].mean():.4f} is implausible")
    if panel["rvol"].min() <= 0:
        problems.append("non-positive realised volatility")

    # Known UK market episodes. The direction of these months is not in doubt,
    # so a wrong sign here means the return series is broken.
    landmarks = {"2008-10": "<0", "2020-03": "<0", "2009-04": ">0", "2022-09": "<0"}
    for ym, direction in landmarks.items():
        per = pd.Period(ym, freq="M")
        if per in panel.index:
            v = panel.loc[per, "eqp"]
            ok = v < 0 if direction == "<0" else v > 0
            if not ok:
                problems.append(f"{ym} premium is {v:+.4f}, expected {direction}")
            elif verbose:
                print(f"[check] {ym} premium {v:+.4%}  ok")

    if problems:
        raise AssertionError("Panel audit failed:\n  - " + "\n  - ".join(problems))
    if verbose:
        print(f"[audit] passed. annualised premium {ann:.2%}, volatility {vol:.2%}")


def descriptive_table(panel: pd.DataFrame,
                      sent: pd.Series | None = None) -> pd.DataFrame:
    """Summary statistics plus first-order autocorrelation, for the data chapter."""
    cols = ["eqp", "dy", "term_spread", "short_rate", "infl", "rvol"]
    df = panel[cols].copy()
    if sent is not None:
        df["sent"] = sent.reindex(df.index)
    rows = []
    for c in df.columns:
        x = df[c].dropna()
        rows.append({
            "variable": c, "n": int(x.size),
            "mean": x.mean(), "sd": x.std(ddof=1),
            "min": x.min(), "p25": x.quantile(0.25), "median": x.median(),
            "p75": x.quantile(0.75), "max": x.max(),
            "skew": x.skew(), "kurtosis": x.kurtosis(),
            "AR1": x.autocorr(1),
        })
    return pd.DataFrame(rows).set_index("variable")

**2.5 Sentiment** &mdash; Sentiment. FinBERT and the Loughran-McDonald dictionary, sharing one article-level aggregation rule so the comparison is clean.

In [ ]:
# Two scorers, sharing one article-level aggregation rule so the comparison
# between them is clean:
#
#     article score = share of positive passages - share of negative passages
#     monthly index = mean article score across articles published that month
#
# sent_t is built from articles published DURING month t and is used to
# forecast the premium in month t+1, so it contains no future information.

_TOKEN_RE = re.compile(r"[a-z']+")
_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9])")
_WS_RE = re.compile(r"\s+")
_BOILER_RE = re.compile(
    r"(\(c\)\s*\d{0,4}[^.]{0,80}(all rights reserved|reserved)\.?"
    r"|copyright\s+\d{4}[^.]{0,80}\.?"
    r"|photograph:\s*[^.]{0,60}\.?)", re.I)


def clean_text(text) -> str:
    t = _BOILER_RE.sub(" ", str(text))
    return _WS_RE.sub(" ", t).strip()


def split_passages(text: str,
                   max_passages: int = MAX_PASSAGES_PER_ARTICLE,
                   min_chars: int = 30,
                   max_chars: int = 1000) -> list:
    """
    Sentence-level split, keeping passages long enough to carry tone.  The
    first `max_passages` are taken rather than a random subset, because news
    convention puts the substance in the opening paragraphs and taking a fixed
    prefix keeps the rule deterministic and reproducible.
    """
    parts = _SENT_SPLIT_RE.split(text)
    out = []
    for p in parts:
        p = p.strip()
        if min_chars <= len(p) <= max_chars:
            out.append(p)
            if len(out) >= max_passages:
                break
    if not out and text.strip():
        out = [text.strip()[:max_chars]]
    return out


# ----------------------------------------------------------------- dictionary
def load_lm_words(path=None) -> tuple:
    """
    Loughran and McDonald (2011) financial sentiment word lists.
    Reads `lm_words.json` ({"positive": [...], "negative": [...]}) if present,
    otherwise falls back to the pysentiment2 package, otherwise raises.
    The lists are fully inflected, so no stemming is required.
    """
    cands = [Path(path)] if path else []
    cands += [DATA_DIR / "lm_words.json", Path("lm_words.json"),
              OUT_DIR / "lm_words.json"]
    for c in cands:
        if c and c.exists():
            d = json.loads(c.read_text())
            return set(d["positive"]), set(d["negative"])
    try:
        import pysentiment2  # noqa
        import pandas as _pd
        p = (Path(pysentiment2.__file__).parent / "static" / "LM.csv")
        d = _pd.read_csv(p)
        return (set(d.loc[d.Positive > 0, "Word"].str.lower()),
                set(d.loc[d.Negative > 0, "Word"].str.lower()))
    except Exception as exc:
        raise FileNotFoundError(
            "Loughran-McDonald word lists not found. Either place lm_words.json "
            "beside the data, or `pip install pysentiment2`."
        ) from exc


def lm_passage_labels(passages, pos: set, neg: set) -> np.ndarray:
    """
    Label each passage positive / negative / neutral by net dictionary count,
    with simple negation handling so 'not a bad quarter' is not scored
    negative.  Negation flips the polarity of the next three tokens, which is
    the standard window in the accounting literature.
    """
    NEGATORS = {"not", "no", "never", "none", "nor", "neither", "without",
                "hardly", "scarcely", "barely", "cannot", "n't", "isn't",
                "wasn't", "aren't", "weren't", "don't", "doesn't", "didn't"}
    labels = []
    for p in passages:
        toks = _TOKEN_RE.findall(p.lower())
        npos = nneg = 0
        flip = 0
        for tk in toks:
            if tk in NEGATORS:
                flip = 3
                continue
            is_p, is_n = tk in pos, tk in neg
            if is_p or is_n:
                if flip > 0:
                    is_p, is_n = is_n, is_p
                npos += int(is_p)
                nneg += int(is_n)
            flip = max(flip - 1, 0)
        labels.append("positive" if npos > nneg else
                      "negative" if nneg > npos else "neutral")
    return np.array(labels)


def article_score_from_labels(labels: np.ndarray) -> float:
    if labels.size == 0:
        return np.nan
    return float((labels == "positive").mean() - (labels == "negative").mean())


def score_corpus_dictionary(articles_csv=None,
                            out_csv=None,
                            checkpoint_dir=None,
                            chunk_rows: int = 20000,
                            time_budget_s=None,
                            verbose: bool = True) -> pd.DataFrame:
    """
    Stream the corpus and build a monthly Loughran-McDonald sentiment index.

    Chunked streaming keeps peak memory at a few hundred megabytes even though
    the corpus file is ~700 MB, which matters on a laptop.  Checkpointing and
    the optional `time_budget_s` mean the job can be stopped and restarted
    without losing work; see `score_corpus_finbert` for why that matters.
    """
    pos, neg = load_lm_words()
    if verbose:
        print(f"[lm] {len(pos)} positive and {len(neg)} negative terms")

    def scorer(chunk):
        out = np.full(len(chunk), np.nan)
        for j, (head, body) in enumerate(zip(chunk["headline"], chunk["body"])):
            text = clean_text((head or "") + ". " + (body or ""))
            labels = lm_passage_labels(split_passages(text), pos, neg)
            out[j] = article_score_from_labels(labels)
        return out

    return _score_corpus(scorer, articles_csv, out_csv,
                         checkpoint_dir or (OUT_DIR / "lm_parts"),
                         chunk_rows, time_budget_s, "lm", verbose)


def _score_corpus(scorer, articles_csv, out_csv, checkpoint_dir,
                  chunk_rows, time_budget_s, tag, verbose):
    """
    Shared driver for both sentiment scorers.

    Article-level scores are flushed to `checkpoint_dir/part_XXXXX.csv` every
    `chunk_rows` articles, and completed parts are skipped on a restart.  Set
    `time_budget_s` to stop cleanly after roughly that many seconds; call the
    function again to carry on from where it stopped.  This is what makes the
    job survive a Colab disconnect, a closed laptop lid, or a sandbox that
    kills long-running background processes.
    """
    import time
    src = Path(articles_csv) if articles_csv else _require(DATA_DIR / RAW["articles"])
    ck = Path(checkpoint_dir)
    ck.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    stopped_early = False
    done_here = 0

    reader = pd.read_csv(src, chunksize=chunk_rows, dtype=str,
                         encoding="utf-8", encoding_errors="replace",
                         engine="c", on_bad_lines="skip")
    for part_no, chunk in enumerate(reader):
        fp = ck / f"part_{part_no:05d}.csv"
        if fp.exists():
            continue
        # At least one part is always completed per invocation, so a budget set
        # too low still makes progress instead of looping forever on nothing.
        if (done_here and time_budget_s is not None
                and (time.time() - t0) > time_budget_s):
            stopped_early = True
            if verbose:
                print(f"[{tag}] time budget reached at part {part_no}; "
                      f"re-run to continue", flush=True)
            break
        chunk = _prep_articles(chunk)
        scores = scorer(chunk)
        # Written to a temporary name first so an interrupted write can never
        # leave a truncated part that a later run would trust and skip.
        tmp = fp.with_suffix(".tmp")
        pd.DataFrame({"month": chunk["month"].astype(str),
                      "article_sent": scores}).to_csv(tmp, index=False)
        tmp.replace(fp)
        done_here += 1
        if verbose:
            print(f"[{tag}] part {part_no}: {len(chunk)} articles "
                  f"({(part_no + 1) * chunk_rows:,} read)", flush=True)

    parts = sorted(ck.glob("part_*.csv"))
    if not parts:
        raise RuntimeError(f"No checkpoint parts written in {ck}.")
    allp = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
    allp = allp.dropna(subset=["article_sent"])
    agg = {}
    for per, s in zip(allp["month"], allp["article_sent"]):
        a = agg.setdefault(pd.Period(per, freq="M"), [0.0, 0])
        a[0] += float(s)
        a[1] += 1
    out = _finalise_monthly(agg)
    out.attrs["complete"] = not stopped_early
    if verbose:
        print(f"[{tag}] {len(parts)} parts, {len(out)} months, "
              f"{int(out['n_articles'].sum()):,} articles"
              f"{'  (INCOMPLETE)' if stopped_early else ''}")
    if out_csv and not stopped_early:
        _write_monthly(out, Path(out_csv))
    return out


# ------------------------------------------------------------------- FinBERT
def score_corpus_finbert(articles_csv=None,
                         out_csv=None,
                         checkpoint_dir=None,
                         chunk_rows: int = CHUNK_ROWS,
                         model_name: str = FINBERT_MODEL,
                         batch_size: int = FINBERT_BATCH,
                         max_len: int = FINBERT_MAX_LEN,
                         time_budget_s=None,
                         verbose: bool = True) -> pd.DataFrame:
    """
    Score the whole corpus with FinBERT and build the monthly index.

    Resumable by design.  Every `chunk_rows` articles the article-level scores
    are flushed to `checkpoint_dir/part_XXXXX.csv`; on restart, completed parts
    are skipped.  A free Colab session will usually disconnect at least once
    during a run of this size, and without checkpointing that means starting
    from zero.

    Run this on a GPU.  On CPU it is roughly fifty times slower.
    """
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    mdl.to(device).eval()

    # Read the label order from the model config rather than assuming it.
    # FinBERT's own ordering is [positive, negative, neutral], which is NOT
    # the alphabetical order a reader would guess.
    id2 = {int(k): str(v).lower() for k, v in mdl.config.id2label.items()}
    label_order = [id2[i] for i in range(mdl.config.num_labels)]
    if set(label_order) != {"positive", "negative", "neutral"}:
        raise ValueError(f"Unexpected FinBERT labels: {label_order}")
    if verbose:
        print(f"[finbert] device={device} labels={label_order}")
        if device == "cpu":
            print("[finbert] WARNING: no GPU detected. Enable one via "
                  "Runtime > Change runtime type in Colab.")

    label_arr = np.array(label_order)

    @torch.no_grad()
    def batch_labels(passages):
        """Argmax label for each passage. Hard classification keeps the
        article score interpretable and bounded in [-1, 1]."""
        out = []
        for i in range(0, len(passages), batch_size):
            enc = tok(passages[i:i + batch_size], padding=True, truncation=True,
                      max_length=max_len, return_tensors="pt").to(device)
            logits = mdl(**enc).logits
            out.append(logits.argmax(dim=-1).cpu().numpy())
        return label_arr[np.concatenate(out)] if out else np.empty(0, dtype=object)

    def scorer(chunk):
        # Every passage in the chunk is flattened into one list so the GPU
        # always sees full batches. Scoring article by article leaves most of
        # the device idle and is several times slower.
        flat, owner = [], []
        for j, (head, body) in enumerate(zip(chunk["headline"], chunk["body"])):
            ps = split_passages(clean_text((head or "") + ". " + (body or "")))
            flat.extend(ps)
            owner.extend([j] * len(ps))
        scores = np.full(len(chunk), np.nan)
        if not flat:
            return scores
        labels = batch_labels(flat)
        owner = np.asarray(owner)
        for j in np.unique(owner):
            scores[j] = article_score_from_labels(labels[owner == j])
        return scores

    return _score_corpus(scorer, articles_csv, out_csv,
                         checkpoint_dir or (OUT_DIR / "finbert_parts"),
                         chunk_rows, time_budget_s, "finbert", verbose)


def _prep_articles(chunk: pd.DataFrame) -> pd.DataFrame:
    """Normalise a raw corpus chunk: parse dates, drop unusable rows."""
    need = {"date", "headline", "body"}
    if not need.issubset(chunk.columns):
        raise ValueError(
            f"Corpus must have columns {sorted(need)}; found {chunk.columns.tolist()}")
    chunk = chunk.copy()
    d = pd.to_datetime(chunk["date"], errors="coerce", format="mixed")
    chunk = chunk.loc[d.notna()].copy()
    chunk["month"] = to_month(d.loc[d.notna()])
    txt = chunk["headline"].fillna("").astype(str) + chunk["body"].fillna("").astype(str)
    return chunk.loc[txt.str.len() >= 40].reset_index(drop=True)


def _finalise_monthly(agg: dict) -> pd.DataFrame:
    if not agg:
        raise RuntimeError("No articles were scored.")
    idx = pd.PeriodIndex(sorted(agg.keys()), freq="M")
    out = pd.DataFrame({
        "sent": [agg[p][0] / agg[p][1] for p in idx],
        "n_articles": [agg[p][1] for p in idx],
    }, index=idx)
    out.index.name = "month"
    return out


def _write_monthly(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    o = df.copy()
    o.insert(0, "date", month_end_ts(o.index).strftime("%Y-%m-%d"))
    o.to_csv(path, index=False)


def read_monthly_sentiment(path) -> pd.Series:
    """Read a monthly sentiment CSV written by either scorer."""
    df = pd.read_csv(path)
    key = "date" if "date" in df.columns else "month"
    s = pd.Series(df["sent"].astype(float).values,
                  index=pd.PeriodIndex(pd.to_datetime(df[key]), freq="M"),
                  name="sent").sort_index()
    return s

**2.6 Forecasting** &mdash; Forecasting. The timing rule in `run_forecasts` is the single most important block of code in the study.

In [ ]:
def _make_model(name: str):
    from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
    from sklearn.linear_model import ElasticNet, LinearRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    if name == "ols":
        pipe = Pipeline([("sc", StandardScaler()), ("m", LinearRegression())])
        grid = {}                      # nothing to tune
    elif name == "enet":
        pipe = Pipeline([("sc", StandardScaler()),
                         ("m", ElasticNet(max_iter=50000, random_state=SEED))])
        # The premium has a standard deviation of about 0.04, so the
        # informative penalty range sits far below sklearn's default alpha=1.0.
        # The grid is nevertheless carried all the way up to 1.0: if the data
        # want total shrinkage, the model should be allowed to collapse onto
        # the historical mean rather than be held back by a binding grid edge.
        # l1_ratio starts at 0.05 (near-ridge) so the sentiment coefficient is
        # not automatically zeroed by the L1 penalty, which would make the
        # central nested comparison degenerate.
        grid = {"m__alpha": list(np.logspace(-6, 0, 13)),
                "m__l1_ratio": [0.05, 0.5, 0.95]}
    elif name == "rf":
        pipe = Pipeline([("m", RandomForestRegressor(
            n_estimators=300, random_state=SEED, n_jobs=1))])
        # Shallow trees and large leaves. Monthly returns have a very low
        # signal-to-noise ratio, so heavy regularisation is essential.
        grid = {"m__max_depth": [2, 3], "m__min_samples_leaf": [10, 25],
                "m__max_features": [0.5, 1.0]}
    elif name == "gbrt":
        pipe = Pipeline([("m", GradientBoostingRegressor(random_state=SEED))])
        grid = {"m__n_estimators": [100, 300], "m__learning_rate": [0.01, 0.05],
                "m__max_depth": [1, 2], "m__subsample": [0.7]}
    else:
        raise ValueError(f"Unknown model: {name}")
    return pipe, grid


def _tune(name: str, X: np.ndarray, y: np.ndarray):
    """Tune inside the training window only, respecting time order."""
    from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
    pipe, grid = _make_model(name)
    if not grid:                       # nothing to tune, e.g. OLS
        pipe.fit(X, y)
        return pipe, {}
    folds = int(min(CV_FOLDS, max(2, len(X) // 30)))
    gs = GridSearchCV(pipe, grid, cv=TimeSeriesSplit(n_splits=folds),
                      scoring="neg_mean_squared_error", n_jobs=-1)
    gs.fit(X, y)
    return gs.best_estimator_, gs.best_params_


def prepare_design(panel: pd.DataFrame, sent: pd.Series | None) -> pd.DataFrame:
    """
    Attach sentiment and build the (predictors at t, premium at t+1) design.
    """
    df = panel.copy()
    if sent is not None:
        df[SENTIMENT_COL] = sent.reindex(df.index)
    have_sent = SENTIMENT_COL in df.columns and df[SENTIMENT_COL].notna().any()
    df["y_next"] = df[TARGET].shift(-1)
    need = (AUGMENTED_PREDICTORS if have_sent else BASELINE_PREDICTORS) + ["y_next"]
    return df.dropna(subset=need)


def run_forecasts(panel: pd.DataFrame,
                  sent: pd.Series | None = None,
                  oos_start: str = OOS_START,
                  models=None,
                  retune_every: int = RETUNE_EVERY,
                  impose_ct: bool = IMPOSE_CT_RESTRICTION,
                  ct_on_bench: bool = CT_ALSO_ON_BENCHMARK,
                  verbose: bool = True):
    """
    Expanding-window out-of-sample forecasts.

    THE TIMING RULE, which is where this kind of study most often goes wrong
    -----------------------------------------------------------------------
    Standing at the close of month t (row i), the forecaster knows the
    predictors for every month up to and including t, and the realised premium
    for every month up to and including t.  A training pair (X_s, y_{s+1})
    therefore requires s+1 <= i, i.e. s <= i-1, so the training set is
    df.iloc[:i] and NOT df.iloc[:i+1].  Including row i would put the premium
    of month t+1 -- the very quantity being forecast -- into the training data.
    The benchmark is the historical mean of the premium through month t.
    """
    models = list(models or MODELS)
    df = prepare_design(panel, sent)
    has_sent = SENTIMENT_COL in df.columns and df[SENTIMENT_COL].notna().all()
    specs = {"base": BASELINE_PREDICTORS}
    if has_sent:
        specs["aug"] = AUGMENTED_PREDICTORS

    periods = df.index
    start = pd.Period(oos_start, freq="M")
    oos_idx = np.where(periods >= start)[0]
    oos_idx = oos_idx[oos_idx >= 1]
    if oos_idx.size == 0:
        raise ValueError(f"No out-of-sample months at or after {oos_start}.")
    if oos_idx[0] < 60:
        raise ValueError(f"Only {oos_idx[0]} training months. Push oos_start later.")

    if verbose:
        print(f"[forecast] {len(df)} usable months | {oos_idx.size} forecasts "
              f"({periods[oos_idx[0]] + 1} to {periods[oos_idx[-1]] + 1}) | "
              f"specs={list(specs)}")

    rows, params_log = [], []
    fitted = {}
    for step, i in enumerate(oos_idx):
        train = df.iloc[:i]                     # see the timing rule above
        y_tr = train["y_next"].to_numpy(dtype=float)
        row = {
            "forecast_for": periods[i] + 1,
            "info_through": periods[i],
            "actual": float(df["y_next"].iloc[i]),
            "bench": float(df[TARGET].iloc[:i + 1].mean()),
            "n_train": int(len(train)),
        }
        retune = (step % retune_every == 0)
        for spec, cols in specs.items():
            X_tr = train[cols].to_numpy(dtype=float)
            X_te = df[cols].iloc[[i]].to_numpy(dtype=float)
            for name in models:
                key = (name, spec)
                if retune or key not in fitted:
                    est, best = _tune(name, X_tr, y_tr)
                    fitted[key] = est
                    params_log.append({"month": str(periods[i]), "model": name,
                                       "spec": spec, **best})
                # Hyper-parameters are held between re-tuning dates but
                # coefficients are re-estimated on the expanded window each month.
                est = fitted[key]
                est.fit(X_tr, y_tr)
                row[f"{name}_{spec}"] = float(est.predict(X_te)[0])
        rows.append(row)
        if verbose and step % 24 == 0:
            print(f"    {periods[i]}  ({step + 1}/{oos_idx.size})", flush=True)

    fc = pd.DataFrame(rows).set_index("forecast_for")
    fc.index = pd.PeriodIndex(fc.index, freq="M")

    if ADD_COMBINATION and len(models) > 1:
        for spec in specs:
            fc[f"combo_{spec}"] = fc[[f"{m}_{spec}" for m in models]].mean(axis=1)

    if impose_ct:
        mcols = [c for c in fc.columns if c.endswith(("_base", "_aug"))]
        fc[mcols] = fc[mcols].clip(lower=0.0)
        if ct_on_bench:
            fc["bench"] = fc["bench"].clip(lower=0.0)

    return fc, pd.DataFrame(params_log)


def model_names(fc: pd.DataFrame) -> list:
    """Model stems present in a forecast frame, in a stable order."""
    stems = [c[:-5] for c in fc.columns if c.endswith("_base")]
    order = ["ols", "enet", "rf", "gbrt", "combo"]
    return [s for s in order if s in stems] + [s for s in stems if s not in order]

**2.7 Statistical Evaluation** &mdash; Statistical evaluation: Campbell-Thompson out-of-sample R-squared and the Clark-West test for nested models.

In [ ]:
def oos_r2(actual, f_large, f_small) -> float:
    """
    Campbell and Thompson (2008) out-of-sample R-squared: the proportional
    reduction in mean squared forecast error of the larger model relative to
    the smaller, nested one.  Positive means the larger model forecasts better.
    """
    a = np.asarray(actual, dtype=float)
    sse_l = np.sum((a - np.asarray(f_large, dtype=float)) ** 2)
    sse_s = np.sum((a - np.asarray(f_small, dtype=float)) ** 2)
    return float(1.0 - sse_l / sse_s)


def clark_west(actual, f_large, f_small, hac_lags: int = 0):
    """
    Clark and West (2007) MSPE-adjusted statistic for nested models.

        f_t = (a - s)^2 - [ (a - l)^2 - (s - l)^2 ]

    H0: the two models forecast equally well.  H1 (one-sided): the larger
    model is better.  For one-step-ahead forecasts the adjusted series is
    serially uncorrelated under the null, so hac_lags=0 is correct; a positive
    value is available as a robustness option.
    """
    a = np.asarray(actual, dtype=float)
    l = np.asarray(f_large, dtype=float)
    s = np.asarray(f_small, dtype=float)
    f = (a - s) ** 2 - ((a - l) ** 2 - (s - l) ** 2)

    # If the larger model has collapsed onto the smaller one, f is identically
    # zero and the statistic is undefined. Report that instead of crashing.
    if np.allclose(f, 0.0, atol=1e-15):
        return np.nan, np.nan

    n = f.size
    e = f - f.mean()
    # Regressing f on a constant makes the t-statistic mean / standard error,
    # so it is computed directly. This avoids a statsmodels dependency and is
    # numerically identical to OLS with HAC covariance.
    gamma0 = float(e @ e) / n
    var = gamma0
    if hac_lags and hac_lags > 0:
        for k in range(1, int(hac_lags) + 1):
            gk = float(e[k:] @ e[:-k]) / n
            var += 2.0 * (1.0 - k / (hac_lags + 1.0)) * gk    # Bartlett kernel
        var = max(var, 1e-30)
    else:
        var = gamma0 * n / (n - 1)                            # small-sample OLS
    se = np.sqrt(var / n)
    t = float(f.mean() / se)
    return t, float(_norm_sf(t))


def _norm_sf(x: float) -> float:
    """Upper-tail standard normal probability, via the error function."""
    import math
    return 0.5 * math.erfc(x / math.sqrt(2.0))


def stars(p) -> str:
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""


def statistical_table(fc: pd.DataFrame, hac_lags: int = 0) -> pd.DataFrame:
    """
    Two blocks.

      vs historical mean      does each model beat the Welch-Goyal benchmark?
      augmented vs baseline   THE CENTRAL TEST. Does sentiment add anything
                              once the traditional predictors are in?
    """
    a = fc["actual"].to_numpy(dtype=float)
    stems = model_names(fc)
    specs = ["base"] + (["aug"] if any(c.endswith("_aug") for c in fc.columns) else [])
    rows = []
    for spec in specs:
        for m in stems:
            f = fc[f"{m}_{spec}"].to_numpy(dtype=float)
            t, p = clark_west(a, f, fc["bench"].to_numpy(dtype=float), hac_lags)
            rows.append({"comparison": "vs historical mean", "model": m, "spec": spec,
                         "oos_r2_pct": 100 * oos_r2(a, f, fc["bench"]),
                         "cw_t": t, "cw_p": p, "sig": stars(p)})
    if "aug" in specs:
        for m in stems:
            big = fc[f"{m}_aug"].to_numpy(dtype=float)
            small = fc[f"{m}_base"].to_numpy(dtype=float)
            t, p = clark_west(a, big, small, hac_lags)
            rows.append({"comparison": "augmented vs baseline", "model": m,
                         "spec": "aug-vs-base",
                         "oos_r2_pct": 100 * oos_r2(a, big, small),
                         "cw_t": t, "cw_p": p, "sig": stars(p)})
    return pd.DataFrame(rows)

**2.8 Economic Evaluation** &mdash; Economic evaluation: mean-variance market timing, certainty equivalents, Sharpe ratios and transaction costs.

In [ ]:
def variance_forecast(panel: pd.DataFrame, target_index: pd.PeriodIndex,
                      window: int = VOL_WINDOW,
                      min_periods: int = VOL_MIN_PERIODS) -> np.ndarray:
    """
    Rolling variance of the realised premium, using only months strictly
    before the target month.  Any remaining gap is filled by an EXPANDING
    mean, never by a full-sample mean, which would leak the future.
    """
    v = panel[TARGET].rolling(window, min_periods=min_periods).var(ddof=1).shift(1)
    v = v.reindex(target_index)
    fallback = panel[TARGET].expanding(min_periods=12).var(ddof=1).shift(1).reindex(target_index)
    v = v.where(v.notna(), fallback)
    return v.ffill().to_numpy(dtype=float)


def backtest(forecast, actual_eqp, rf, var_hat, cost=TRANSACTION_COST,
             gamma=GAMMA, w_min=W_MIN, w_max=W_MAX):
    """
    Mean-variance market timing.  Each month the investor holds

        w_t = (1 / gamma) * E_t[premium] / Var_t[premium],  clipped to [w_min, w_max]

    in equities and the remainder at the risk-free rate.  Transaction costs
    are charged on the traded amount |w_t - w_{t-1,end}|, where w_{t-1,end} is
    the weight AFTER the previous month's returns have moved it.  Ignoring
    that drift overstates turnover and therefore overstates costs.
    """
    f = np.asarray(forecast, dtype=float)
    a = np.asarray(actual_eqp, dtype=float)
    rf = np.asarray(rf, dtype=float)
    v = np.asarray(var_hat, dtype=float)

    w = np.clip((1.0 / gamma) * f / v, w_min, w_max)
    gross = np.empty_like(w)
    net = np.empty_like(w)
    turn = np.empty_like(w)
    w_end = 0.0
    for t in range(w.size):
        r_eq = a[t] + rf[t]
        r_p = rf[t] + w[t] * a[t]
        trade = abs(w[t] - w_end)
        gross[t] = r_p
        net[t] = r_p - cost * trade
        turn[t] = trade
        denom = 1.0 + r_p
        w_end = (w[t] * (1.0 + r_eq)) / denom if denom > 1e-12 else w[t]
    return gross, net, turn, w


def cer(returns, rf, gamma=GAMMA) -> float:
    """Annualised certainty-equivalent return of a mean-variance investor."""
    ex = np.asarray(returns, dtype=float) - np.asarray(rf, dtype=float)
    return float(12.0 * (np.mean(ex) - 0.5 * gamma * np.var(ex, ddof=1)))


def sharpe(returns, rf) -> float:
    ex = np.asarray(returns, dtype=float) - np.asarray(rf, dtype=float)
    sd = np.std(ex, ddof=1)
    return float(np.sqrt(12.0) * np.mean(ex) / sd) if sd > 0 else np.nan


def max_drawdown(returns) -> float:
    w = np.cumprod(1.0 + np.asarray(returns, dtype=float))
    return float((w / np.maximum.accumulate(w) - 1.0).min())


def economic_table(fc: pd.DataFrame, panel: pd.DataFrame,
                   cost=TRANSACTION_COST, gamma=GAMMA,
                   w_max=W_MAX) -> pd.DataFrame:
    a = fc["actual"].to_numpy(dtype=float)
    rf = panel["rf"].reindex(fc.index).ffill().to_numpy(dtype=float)
    var_hat = variance_forecast(panel, fc.index)

    stems = model_names(fc)
    specs = ["base"] + (["aug"] if any(c.endswith("_aug") for c in fc.columns) else [])
    cols = ["bench"] + [f"{m}_{s}" for s in specs for m in stems]

    rows = []
    for col in cols:
        g, n, to, w = backtest(fc[col].to_numpy(dtype=float), a, rf, var_hat,
                               cost=cost, gamma=gamma, w_max=w_max)
        rows.append({"strategy": col,
                     "CER_gross_pct": 100 * cer(g, rf, gamma),
                     "CER_net_pct": 100 * cer(n, rf, gamma),
                     "Sharpe_gross": sharpe(g, rf), "Sharpe_net": sharpe(n, rf),
                     "ann_ret_net_pct": 100 * np.mean(n) * 12,
                     "ann_vol_pct": 100 * np.std(n, ddof=1) * np.sqrt(12),
                     "max_dd_pct": 100 * max_drawdown(n),
                     "avg_turnover": float(np.mean(to)),
                     "avg_weight": float(np.mean(w))})

    bh = a + rf
    rows.append({"strategy": "buy_and_hold",
                 "CER_gross_pct": 100 * cer(bh, rf, gamma),
                 "CER_net_pct": 100 * cer(bh, rf, gamma),
                 "Sharpe_gross": sharpe(bh, rf), "Sharpe_net": sharpe(bh, rf),
                 "ann_ret_net_pct": 100 * np.mean(bh) * 12,
                 "ann_vol_pct": 100 * np.std(bh, ddof=1) * np.sqrt(12),
                 "max_dd_pct": 100 * max_drawdown(bh),
                 "avg_turnover": 0.0, "avg_weight": 1.0})

    out = pd.DataFrame(rows)
    base = out.loc[out.strategy == "bench", "CER_net_pct"].iloc[0]
    out["CER_gain_vs_bench_pct"] = out["CER_net_pct"] - base
    return out

**2.9 Robustness** &mdash; Robustness: sub-periods, cost and risk-aversion grids, and the look-ahead test.

In [ ]:
def subperiod_table(fc: pd.DataFrame, cuts=("2011-01", "2020-01", "2026-01"),
                    hac_lags: int = 0) -> pd.DataFrame:
    """Split the out-of-sample period so a single episode cannot drive everything."""
    rows = []
    bounds = [pd.Period(c, freq="M") for c in cuts]
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        sub = fc.loc[(fc.index >= lo) & (fc.index < hi)]
        if len(sub) < 12:
            continue
        st = statistical_table(sub, hac_lags=hac_lags)
        st.insert(0, "period", f"{sub.index.min()} to {sub.index.max()}")
        st.insert(1, "n", len(sub))
        rows.append(st)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def cost_gamma_grid(fc: pd.DataFrame, panel: pd.DataFrame,
                    costs=(0.0, 0.0005, 0.0010, 0.0025),
                    gammas=(2.0, 3.0, 5.0)) -> pd.DataFrame:
    """CER gain over the benchmark strategy across trading costs and risk aversion."""
    rows = []
    for c in costs:
        for g in gammas:
            t = economic_table(fc, panel, cost=c, gamma=g)
            base = t.loc[t.strategy == "bench", "CER_net_pct"].iloc[0]
            for _, r in t.iterrows():
                if r.strategy in ("bench", "buy_and_hold"):
                    continue
                rows.append({"cost_bps": 1e4 * c, "gamma": g,
                             "strategy": r.strategy,
                             "CER_net_pct": r.CER_net_pct,
                             "CER_gain_vs_bench_pct": r.CER_net_pct - base,
                             "Sharpe_net": r.Sharpe_net})
    return pd.DataFrame(rows)


def no_lookahead_test(panel: pd.DataFrame, sent: pd.Series | None,
                      oos_start: str = OOS_START, verbose: bool = True):
    """
    Truncate the sample immediately after a given forecast and re-run.  If the
    forecasts genuinely use only past information, deleting everything that
    comes after them must leave them numerically unchanged.

    Note on the earlier version of this test: keeping an extra month of data
    past the cut point preserves exactly the leak the test is supposed to
    detect, so the test passes whether or not the code is correct.  Here the
    cut keeps one extra month solely so the FINAL forecast still has a
    realised target, and the comparison then excludes that final forecast.
    """
    full, _ = run_forecasts(panel, sent, oos_start=oos_start, verbose=False)
    cut = full.index[len(full) // 2]                    # a target month
    keep = panel.index <= cut
    p_trunc = panel.loc[keep]
    s_trunc = sent.loc[sent.index <= cut] if sent is not None else None

    trunc, _ = run_forecasts(p_trunc, s_trunc, oos_start=oos_start, verbose=False)
    cols = [c for c in full.columns
            if c.endswith(("_base", "_aug")) or c == "bench"]
    common = full.index.intersection(trunc.index)
    common = common[common < trunc.index.max()]         # drop the truncated edge
    diff = (full.loc[common, cols] - trunc.loc[common, cols]).abs()
    worst = float(diff.to_numpy().max())
    ok = worst < 1e-10
    if verbose:
        print(f"[lookahead] cut after {cut}; compared {len(common)} forecasts "
              f"x {len(cols)} series")
        print(f"[lookahead] largest absolute discrepancy: {worst:.3e}")
        print("[lookahead] PASS, no future information is used." if ok else
              "[lookahead] FAIL\n" + diff.max().sort_values(ascending=False).head().to_string())
    return ok, worst

**2.10 Figures** &mdash; Figures.

In [ ]:
def _plt():
    """
    Return pyplot, forcing the non-interactive Agg backend ONLY when we are not
    inside IPython.  Hard-coding Agg in a notebook would silently disable
    inline figures for everything the user plots afterwards.
    """
    import matplotlib
    try:
        get_ipython  # noqa: F821  provided by IPython
        in_ipython = True
    except NameError:
        in_ipython = False
    if not in_ipython:
        try:
            import IPython
            in_ipython = IPython.get_ipython() is not None
        except Exception:
            in_ipython = False
    if not in_ipython:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    return plt



def figure_cumulative_sse(fc: pd.DataFrame, path) -> None:
    """
    Goyal-Welch cumulative SSE difference: benchmark minus model.  An upward
    slope means the model is beating the historical mean over that stretch,
    which exposes predictability that is concentrated in a few episodes rather
    than being spread through the sample.
    """
    plt = _plt()

    a = fc["actual"].to_numpy(dtype=float)
    b = fc["bench"].to_numpy(dtype=float)
    x = month_end_ts(fc.index)
    stems = model_names(fc)
    has_aug = any(c.endswith("_aug") for c in fc.columns)

    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    for m in stems:
        d = np.cumsum((a - b) ** 2 - (a - fc[f"{m}_base"].to_numpy(dtype=float)) ** 2)
        ax.plot(x, d, "--", lw=1.2, alpha=0.75, label=f"{m} baseline")
    if has_aug:
        for m in stems:
            d = np.cumsum((a - b) ** 2 - (a - fc[f"{m}_aug"].to_numpy(dtype=float)) ** 2)
            ax.plot(x, d, "-", lw=1.6, label=f"{m} augmented")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylabel("Cumulative SSE: benchmark minus model")
    ax.set_xlabel("")
    ax.set_title("Out-of-sample performance relative to the historical mean")
    ax.legend(fontsize=8, ncol=2, frameon=False)
    ax.grid(alpha=0.25, lw=0.5)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def figure_series(panel: pd.DataFrame, sent: pd.Series | None, path) -> None:
    """The premium, the sentiment index and the two most-watched predictors."""
    plt = _plt()

    x = month_end_ts(panel.index)
    n = 4 if sent is not None else 3
    fig, axes = plt.subplots(n, 1, figsize=(9.5, 2.2 * n), sharex=True)
    axes[0].plot(x, 100 * panel["eqp"], lw=0.9, color="#1f3a68")
    axes[0].axhline(0, color="k", lw=0.6)
    axes[0].set_ylabel("premium, %")
    axes[1].plot(x, 100 * panel["dy"], lw=1.2, color="#1f3a68")
    axes[1].set_ylabel("dividend yield, %")
    axes[2].plot(x, 100 * panel["term_spread"], lw=1.2, color="#1f3a68")
    axes[2].axhline(0, color="k", lw=0.6)
    axes[2].set_ylabel("term spread, %")
    if sent is not None:
        s = sent.reindex(panel.index)
        axes[3].plot(x, s, lw=1.2, color="#8c2d19")
        axes[3].axhline(float(s.mean()), color="k", lw=0.6, ls=":")
        axes[3].set_ylabel("news sentiment")
    for ax in axes:
        ax.grid(alpha=0.25, lw=0.5)
    axes[0].set_title("UK equity premium, traditional predictors and news sentiment")
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def figure_weights(fc: pd.DataFrame, panel: pd.DataFrame, path,
                   stem: str = "combo") -> None:
    """Equity weight through time for the benchmark and the two specifications."""
    plt = _plt()

    a = fc["actual"].to_numpy(dtype=float)
    rf = panel["rf"].reindex(fc.index).ffill().to_numpy(dtype=float)
    v = variance_forecast(panel, fc.index)
    x = month_end_ts(fc.index)
    stems = model_names(fc)
    if stem not in stems:
        stem = stems[0]

    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    for col, style, lab in [("bench", ":", "benchmark"),
                            (f"{stem}_base", "--", f"{stem} baseline"),
                            (f"{stem}_aug", "-", f"{stem} augmented")]:
        if col not in fc.columns:
            continue
        _, _, _, w = backtest(fc[col].to_numpy(dtype=float), a, rf, v)
        ax.plot(x, w, style, lw=1.3, label=lab)
    ax.set_ylabel("weight in equities")
    ax.set_title("Market-timing allocations")
    ax.legend(fontsize=8, frameon=False)
    ax.grid(alpha=0.25, lw=0.5)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

**2.11 Export** &mdash; Export helpers.

In [ ]:
def _stamp(df: pd.DataFrame) -> pd.DataFrame:
    """Replace a monthly PeriodIndex with an ISO month-end date column."""
    o = df.copy()
    if isinstance(o.index, pd.PeriodIndex):
        o.insert(0, "date", month_end_ts(o.index).strftime("%Y-%m-%d"))
        o = o.reset_index(drop=True)
    return o


def save_all(out_dir=None, **frames) -> list:
    d = Path(out_dir or OUT_DIR)
    d.mkdir(parents=True, exist_ok=True)
    written = []
    for name, obj in frames.items():
        if obj is None:
            continue
        p = d / f"{name}.csv"
        if isinstance(obj, pd.Series):
            obj = obj.to_frame()
        _stamp(obj).to_csv(p, index=False)
        written.append(str(p))
    return written


def show(df: pd.DataFrame, nd: int = 3) -> str:
    with pd.option_context("display.width", 200, "display.max_columns", 60):
        return df.round(nd).to_string(index=False)

In [ ]:
# Bring the module-level names into the notebook namespace under a short alias,
# so the analysis cells below read the same as the command-line script.
import types
U = types.SimpleNamespace(**{k: v for k, v in globals().items()
                             if not k.startswith("__")})
print("library loaded")

---
## 3. The panel

`build_panel` reads all five numerical files, aligns them to month end and
enforces the real-time information rule described in its docstring. Two
alignment choices matter enough to state in the text:

- the risk-free rate subtracted from month *t*'s index return is the SONIA
  fixing at the **end of month t-1**, because that is the rate an investor
  could actually have locked in when the position was opened;
- inflation is **lagged one month**, because ONS publishes month *t*'s CPI in
  the middle of month *t+1*, so the contemporaneous figure is not in the
  month-*t* information set.

`_audit_panel` then hard-checks the result against episodes whose direction is
not in doubt (October 2008, March 2020, April 2009, September 2022). If the
return series were mis-signed or mis-scaled, it raises rather than quietly
producing a wrong dissertation.

In [ ]:
panel = build_panel()
panel.tail(6).round(5)

A first look at the premium itself. Table 5.1, which adds the sentiment
variable, is produced at the end of Section 4 once that series exists.

In [ ]:
print(show(descriptive_table(panel).reset_index(), 4))
print(f"\nannualised mean premium   {panel[TARGET].mean() * 12:7.2%}")
print(f"annualised volatility     {panel[TARGET].std(ddof=1) * (12 ** 0.5):7.2%}")
print(f"months with a negative premium: "
      f"{(panel[TARGET] < 0).mean():.1%}")

---
## 4. Sentiment

Both scorers use the same two-step rule, so any difference between them is
attributable to the classifier and not to the aggregation:

```
article score = share of positive passages - share of negative passages
monthly index = mean article score over articles published that month
```

`sent_t` is built from articles published **during** month *t* and used to
forecast the premium in month *t+1*, so it contains no future information.

### 4a. FinBERT (the main measure)

Run this on a GPU. In Colab: *Runtime > Change runtime type > T4 GPU*.

The cell is **resumable**. Article scores are flushed to
`OUT_DIR/finbert_parts/part_XXXXX.csv` every 2,000 articles and completed parts
are skipped on a restart, so a Colab disconnect costs you one chunk rather than
the whole run. Set `TIME_BUDGET` if you want it to stop cleanly by itself; just
run the cell again to continue.

In [ ]:
TIME_BUDGET = None      # e.g. 3000 to stop after ~50 minutes, then re-run

sent_fb = score_corpus_finbert(
    out_csv=os.path.join(OUT_DIR, "sentiment_monthly_finbert.csv"),
    checkpoint_dir=os.path.join(OUT_DIR, "finbert_parts"),
    time_budget_s=TIME_BUDGET,
)
if sent_fb.attrs.get("complete", True):
    print("\nComplete.")
    print(sent_fb.describe().round(4).to_string())
else:
    print("\nStopped early. Re-run this cell to continue from the last checkpoint.")

### 4b. Loughran-McDonald dictionary (the robustness comparison)

Fast, CPU-only, no model download. Section 2.3 of the dissertation argues that
a transformer should beat a bag-of-words dictionary because it reads context
and negation; this is the cell that lets you test that claim rather than assert
it. The scorer includes a three-token negation window, so *"not a bad quarter"*
is scored positive rather than negative, which is the fairest version of the
dictionary approach to compare against.

In [ ]:
sent_lm = score_corpus_dictionary(
    out_csv=os.path.join(OUT_DIR, "sentiment_monthly_lm.csv"),
    checkpoint_dir=os.path.join(OUT_DIR, "lm_parts"),
)
print(sent_lm.describe().round(4).to_string())

### 4c. Choose the sentiment series for the main results

FinBERT if it exists, otherwise the dictionary. If both exist, the correlation
between them is worth reporting: it tells the reader how much the choice of
classifier actually matters.

In [ ]:
SENT_PATH = None        # set explicitly to override the automatic choice

def pick_sentiment(explicit=None):
    if explicit:
        return read_monthly_sentiment(explicit), os.path.basename(explicit)
    for fn, tag in [("sentiment_monthly_finbert.csv", "FinBERT"),
                    ("sentiment_monthly_lm.csv", "Loughran-McDonald")]:
        for folder in (OUT_DIR, DATA_DIR):
            p = os.path.join(folder, fn)
            if os.path.exists(p):
                return read_monthly_sentiment(p), tag
    return None, "none"

sent, sent_tag = pick_sentiment(SENT_PATH)
print("sentiment source:", sent_tag)
if sent is None:
    print("No sentiment series found. Section 5 will run the baseline only.")
else:
    print(f"{len(sent)} months, {sent.index.min()} to {sent.index.max()}")
    print(f"mean {sent.mean():+.4f}  sd {sent.std():.4f}  AR(1) {sent.autocorr(1):.3f}")
    cov = sent.reindex(panel.index).notna().mean()
    print(f"covers {cov:.1%} of panel months")

    fb = os.path.join(OUT_DIR, "sentiment_monthly_finbert.csv")
    lm = os.path.join(OUT_DIR, "sentiment_monthly_lm.csv")
    if os.path.exists(fb) and os.path.exists(lm):
        a, b = read_monthly_sentiment(fb), read_monthly_sentiment(lm)
        j = pd.concat([a.rename("finbert"), b.rename("lm")], axis=1).dropna()
        print(f"\ncorrelation FinBERT vs dictionary: {j.finbert.corr(j.lm):.3f} "
              f"over {len(j)} months")

### 4d. Raw predictive correlations

Not a test, just orientation before the models run. A near-zero correlation
between sentiment at *t* and the premium at *t+1*, alongside a clearly positive
**contemporaneous** correlation, is the signature of a variable that moves
*with* the market rather than *ahead* of it. If that is what the data show,
expect the formal tests to find nothing, and say so.

In [ ]:
X = panel[BASELINE_PREDICTORS].copy()
if sent is not None:
    X["sent"] = sent.reindex(panel.index)
lag_corr = (pd.concat([X, panel[TARGET].shift(-1).rename("eqp_next")], axis=1)
              .dropna().corr()["eqp_next"].drop("eqp_next"))
print("corr(predictor at t, premium at t+1)")
print(lag_corr.round(4).to_string())
if sent is not None:
    same = pd.concat([X["sent"], panel[TARGET]], axis=1).dropna().corr().iloc[0, 1]
    print(f"\ncorr(sentiment at t, premium at t)  = {same:+.4f}   [contemporaneous]")
    print(f"corr(sentiment at t, premium at t+1) = {lag_corr['sent']:+.4f}   [predictive]")

### Table 5.1 &mdash; Descriptive statistics

Now that the sentiment series exists, this is the table for the data chapter.
`AR1` is first-order autocorrelation: the predictors are all highly persistent,
which is the property that motivates the Kostakis, Magdalinos and
Stamatogiannis (2015) caution against reading too much into in-sample
significance.

In [ ]:
desc = descriptive_table(panel, sent)
print(show(desc.reset_index(), 4))

---
## 5. Out-of-sample forecasts

Expanding window, re-estimated every month. Read the timing rule in
`run_forecasts` before interpreting anything: standing at the close of month
*t*, the training set must end at *t-1*, because a training pair needs the
premium of the month *after* its predictors, and the premium of month *t+1* is
the quantity being forecast. Training on `df.iloc[:i+1]` instead of
`df.iloc[:i]` leaks the target into the training data and inflates every
number downstream.

Four models: OLS, elastic net, random forest, gradient boosting, plus their
equal-weighted combination. Keeping plain OLS alongside the machine-learning
methods is what separates *the predictors contribute* from *flexibility
contributes*.

Runtime is roughly five to ten minutes for a 2011 start on a laptop. Set
`RETUNE_EVERY = 24` to halve it while testing.

In [ ]:
OOS_START_HERE = "2011-01"      # 120 months of initial training data
RETUNE_HERE    = 12             # re-tune hyper-parameters every N months

fc, chosen_params = run_forecasts(panel, sent,
                                  oos_start=OOS_START_HERE,
                                  retune_every=RETUNE_HERE,
                                  impose_ct=True)
print()
fc.head(6).round(5)

---
## 6. Statistical evaluation

### Table 5.2

Two blocks, and the second is the point of the dissertation.

- **vs historical mean** &mdash; does each model beat the Welch-Goyal
  benchmark? This says whether the traditional predictors have any
  out-of-sample value at all.
- **augmented vs baseline** &mdash; the central test. Because the baseline
  predictors are strictly nested inside the augmented set, a positive
  `oos_r2_pct` with a significant Clark-West statistic here means sentiment
  adds forecasting power *beyond* the traditional predictors. Nothing else in
  the table answers the research question.

`cw_p` is one-sided. Stars: \*\*\* 1%, \*\* 5%, \* 10%.

In [ ]:
stat = statistical_table(fc, hac_lags=0)
print(show(stat))

A blank `cw_t` in the augmented-vs-baseline block is informative rather than
broken: it means the larger model's forecasts are numerically identical to the
smaller one's, i.e. the estimator set the sentiment coefficient to exactly zero
in every month. Report that as a finding.

---
## 7. Economic evaluation

### Table 5.3

A forecast can be statistically detectable and still worthless after costs.
Each set of forecasts drives a mean-variance timing rule,
`w = (1/gamma) * forecast / variance`, clipped to [0, 1.5], with 10 bps charged
on the traded amount. Costs are computed on `|w_t - w_{t-1,end}|`, where
`w_{t-1,end}` is last month's weight *after* returns have moved it, since
ignoring that drift overstates turnover and so overstates costs.

`CER_gain_vs_bench_pct` is the number this literature headlines: annualised
certainty-equivalent gain over timing on the historical mean, net of costs.

In [ ]:
econ = economic_table(fc, panel, cost=TRANSACTION_COST, gamma=GAMMA)
print(show(econ))

---
## 8. Robustness

### Table 5.4 &mdash; Sub-periods

Splitting at 2020 checks that nothing is driven by the COVID crash alone. A
result that exists in one sub-period and reverses in the other is not a result.

In [ ]:
sub = subperiod_table(fc, hac_lags=0)
print(show(sub) if len(sub) else "not enough months to split")

### Table 5.5 &mdash; Trading costs and risk aversion

Rows are strategies, columns are (gamma, cost in bps). Entries are annualised
CER gain over the benchmark strategy in percentage points. If the sign flips
between 0 and 25 bps, the economic finding is a cost artefact.

In [ ]:
grid = cost_gamma_grid(fc, panel, costs=(0.0, 0.0005, 0.0010, 0.0025),
                       gammas=(2.0, 3.0, 5.0))
print(grid.pivot_table(index="strategy", columns=["gamma", "cost_bps"],
                       values="CER_gain_vs_bench_pct").round(2).to_string())

### Table 5.6 &mdash; Without the Campbell-Thompson restriction

The zero floor on forecasts materially changes results and a reader will ask,
so report both. This re-runs the whole loop, so it takes as long again as
Section 5.

In [ ]:
fc_noct, _ = run_forecasts(panel, sent, oos_start=OOS_START_HERE,
                           retune_every=RETUNE_HERE, impose_ct=False,
                           verbose=False)
stat_noct = statistical_table(fc_noct, hac_lags=0)
print(show(stat_noct))

### The look-ahead test

Truncate the sample just after a forecast and re-run. If that forecast used
only past information, deleting everything after it must leave it numerically
unchanged. Cite this in the text.

One caveat worth knowing: a version of this test that keeps an extra month of
data past the cut preserves exactly the leak it is meant to detect, and so
passes whether or not the code is correct. Here the extra month exists only so
the final forecast still has a realised target, and that final forecast is
excluded from the comparison.

In [ ]:
ok, worst = no_lookahead_test(panel, sent, oos_start=OOS_START_HERE)
assert ok, "Look-ahead test FAILED. Do not report these results."

### Optional further checks

Each is one line. Run whichever the write-up claims.

In [ ]:
# Bank Rate instead of SONIA as the risk-free proxy
# panel_bank = build_panel(rf_series=RF_SERIES_ALT)

# Contemporaneous CPI, i.e. deliberately reintroduce the publication look-ahead.
# The gap against the main results measures how much that mistake would be worth.
# panel_nolag = build_panel(lag_inflation=False)

# HAC standard errors in the Clark-West test
# print(show(statistical_table(fc, hac_lags=6)))

# Later out-of-sample start
# fc_late, _ = run_forecasts(panel, sent, oos_start="2015-01")
# print(show(statistical_table(fc_late)))

---
## 9. Figures and export

In [ ]:
figure_series(panel, sent, os.path.join(OUT_DIR, "fig_5_1_series.png"))
figure_cumulative_sse(fc, os.path.join(OUT_DIR, "fig_5_2_cumulative_sse.png"))
figure_weights(fc, panel, os.path.join(OUT_DIR, "fig_5_3_weights.png"))

from IPython.display import Image, display
for f in ["fig_5_1_series.png", "fig_5_2_cumulative_sse.png", "fig_5_3_weights.png"]:
    display(Image(filename=os.path.join(OUT_DIR, f)))

In [ ]:
written = save_all(OUT_DIR,
                   panel=panel,
                   table_5_1_descriptives=desc.reset_index(),
                   forecasts=fc,
                   table_5_2_statistical=stat,
                   table_5_3_economic=econ,
                   table_5_4_subperiods=sub,
                   table_5_5_cost_gamma=grid,
                   table_5_6_no_ct=stat_noct,
                   chosen_hyperparams=chosen_params)
for w in written:
    print("wrote", w)

### Headline numbers

The single cell to read off when writing Chapter 5, so the wrong row of the
wrong table cannot end up in the text.

In [ ]:
print("=" * 70)
print("SAMPLE     ", f"{panel.index.min()} to {panel.index.max()}, "
                     f"{len(panel)} months")
print("OUT OF SAMPLE", f"{fc.index.min()} to {fc.index.max()}, "
                       f"{len(fc)} forecasts")
print("SENTIMENT  ", sent_tag)
print("=" * 70)

b = stat[(stat.comparison == "vs historical mean") & (stat.spec == "base")]
top = b.loc[b.oos_r2_pct.idxmax()]
print(f"\nBASELINE vs HISTORICAL MEAN")
print(f"   best R2_OS      {top.oos_r2_pct:+.3f}%  ({top.model})")
print(f"   Clark-West t    {top.cw_t:.2f}   p = {top.cw_p:.3f} {top.sig}")
print(f"   models with R2_OS > 0: {int((b.oos_r2_pct > 0).sum())} of {len(b)}")

a = stat[stat.comparison == "augmented vs baseline"]
if len(a):
    ta = a.loc[a.oos_r2_pct.idxmax()]
    print(f"\nSENTIMENT vs BASELINE  <-- the research question")
    print(f"   best R2_OS      {ta.oos_r2_pct:+.3f}%  ({ta.model})")
    print(f"   Clark-West t    {ta.cw_t:.2f}   p = {ta.cw_p:.3f} {ta.sig}")
    print(f"   models with R2_OS > 0: {int((a.oos_r2_pct > 0).sum())} of {len(a)}")

e = econ[econ.strategy != "buy_and_hold"]
te = e.loc[e.CER_gain_vs_bench_pct.idxmax()]
print(f"\nECONOMIC VALUE (net of {1e4 * TRANSACTION_COST:.0f} bps)")
print(f"   best CER gain over benchmark  {te.CER_gain_vs_bench_pct:+.3f}% p.a. "
      f"({te.strategy})")
print(f"   benchmark net Sharpe          "
      f"{econ.loc[econ.strategy == 'bench', 'Sharpe_net'].iloc[0]:.3f}")
print(f"   buy and hold net Sharpe       "
      f"{econ.loc[econ.strategy == 'buy_and_hold', 'Sharpe_net'].iloc[0]:.3f}")
print(f"\nLOOK-AHEAD TEST  {'PASS' if ok else 'FAIL'}   (max discrepancy {worst:.1e})")
print("=" * 70)

---
## A note on interpreting a null result

If sentiment adds nothing, that is an answer, not a failure, and the
dissertation's introduction already commits to treating it as one. The design
is what makes the null informative: the nesting isolates the marginal
contribution of text, the timing rule rules out look-ahead as an explanation,
the sub-period split rules out a single episode, and the dictionary comparison
rules out "the classifier was too crude". Having closed those doors, "modern
transformer sentiment from a large UK news corpus does not improve
out-of-sample forecasts of the UK equity premium once prices and macro
variables are accounted for" is a finding with content.

The temptation to keep changing the specification until sentiment works is
exactly what Welch and Goyal (2008) were warning about. Do not.